# Funciones · Juegos de tablero

Juegos de tablero diseñados con funciones sobre matrices. Los originales usaban el ratón en PythonG; aquí se juega por coordenadas en modo texto y cada juego se prueba con una partida de ejemplo.

## 1. Memorión con niveles

Tablero de cartas boca abajo con parejas de letras; se destapan dos y, si coinciden, se quedan descubiertas. Ofrecer tres niveles: fácil (3 × 4), normal (4 × 6) y difícil (6 × 8).

In [1]:
import random
from string import ascii_uppercase

NIVELES = {"fácil": (3, 4), "normal": (4, 6), "difícil": (6, 8)}

def crea_tablero(filas, columnas, grupo=2, semilla=None):
    assert filas * columnas % grupo == 0
    rng = random.Random(semilla)
    cartas = [letra for letra in ascii_uppercase[:filas * columnas // grupo] for _ in range(grupo)]
    rng.shuffle(cartas)
    return [cartas[f * columnas:(f + 1) * columnas] for f in range(filas)]

def muestra(tablero, descubiertas):
    for f, fila in enumerate(tablero):
        print(" ".join(c if (f, k) in descubiertas else "?" for k, c in enumerate(fila)))

def jugada(tablero, descubiertas, casillas):
    letras = {tablero[f][c] for f, c in casillas}
    if len(letras) == 1:
        descubiertas.update(casillas)
        return True
    return False

def partida_perfecta(tablero, grupo=2):
    # juega conociendo el tablero para comprobar las reglas
    posiciones = {}
    for f, fila in enumerate(tablero):
        for c, letra in enumerate(fila):
            posiciones.setdefault(letra, []).append((f, c))
    descubiertas, intentos = set(), 0
    for casillas in posiciones.values():
        intentos += 1
        assert jugada(tablero, descubiertas, casillas)
    return intentos, len(descubiertas) == sum(len(f) for f in tablero)

for nivel, (filas, columnas) in NIVELES.items():
    tablero = crea_tablero(filas, columnas, semilla=1)
    print(nivel, partida_perfecta(tablero))
tablero = crea_tablero(3, 4, semilla=1)
descubiertas = set()
print(jugada(tablero, descubiertas, [(0, 0), (0, 1)]))
muestra(tablero, descubiertas)

fácil (6, True)
normal (12, True)
difícil (24, True)
False
? ? ? ?
? ? ? ?
? ? ? ?


## 2. Memorión3

Variante en la que hay que emparejar grupos de 3 letras iguales (el número de casillas debe ser múltiplo de 3).

In [2]:
tablero3 = crea_tablero(3, 4, grupo=3, semilla=2)
for fila in tablero3:
    print(fila)
print(partida_perfecta(tablero3, grupo=3))

['D', 'D', 'B', 'B']
['C', 'C', 'C', 'A']
['B', 'D', 'A', 'A']
(4, True)


## 3. Buscaminas

Matriz de minas colocadas al azar, matriz con el número de minas vecinas (cuidando los bordes), descubrir casillas y marcarlas. La partida se gana cuando las marcas coinciden con las minas. Al descubrir una casilla sin minas vecinas, se descubren todas las alcanzables desde ella.

In [3]:
from collections import deque

def crea_buscaminas(filas, columnas, num_minas, semilla=None):
    rng = random.Random(semilla)
    casillas = rng.sample([(f, c) for f in range(filas) for c in range(columnas)], num_minas)
    minas = [[(f, c) in casillas for c in range(columnas)] for f in range(filas)]
    cuenta = [[sum(minas[f + i][c + j] for i in (-1, 0, 1) for j in (-1, 0, 1)
                   if 0 <= f + i < filas and 0 <= c + j < columnas) for c in range(columnas)] for f in range(filas)]
    return minas, cuenta

def descubre(minas, cuenta, visibles, f, c):
    if minas[f][c]:
        return False
    cola = deque([(f, c)])
    while cola:
        f, c = cola.popleft()
        if (f, c) in visibles:
            continue
        visibles.add((f, c))
        if cuenta[f][c] == 0:
            for i in (-1, 0, 1):
                for j in (-1, 0, 1):
                    if 0 <= f + i < len(minas) and 0 <= c + j < len(minas[0]):
                        cola.append((f + i, c + j))
    return True

def ganada(minas, marcas):
    return marcas == {(f, c) for f, fila in enumerate(minas) for c, m in enumerate(fila) if m}

def muestra_buscaminas(minas, cuenta, visibles, marcas):
    for f in range(len(minas)):
        print(" ".join("F" if (f, c) in marcas else str(cuenta[f][c] or ".") if (f, c) in visibles else "#"
                       for c in range(len(minas[0]))))

minas, cuenta = crea_buscaminas(6, 8, 6, semilla=3)
visibles, marcas = set(), set()
inicio = next((f, c) for f in range(6) for c in range(8) if cuenta[f][c] == 0 and not minas[f][c])
descubre(minas, cuenta, visibles, *inicio)
muestra_buscaminas(minas, cuenta, visibles, marcas)
marcas = {(f, c) for f in range(6) for c in range(8) if minas[f][c]}
print("Ganada:", ganada(minas, marcas))

# 1 . . . . 1 #
# 1 . . . . 2 #
# 1 . . . . 2 #
# 1 1 1 1 2 3 #
# # # # # # # #
# # # # # # # #
Ganada: True


## 4. Tres en raya para dos personas

Dos jugadores alternan `X` y `O` indicando fila y columna; gana quien completa una línea.

In [4]:
LINEAS = [[(f, c) for c in range(3)] for f in range(3)] + [[(f, c) for f in range(3)] for c in range(3)] + \
         [[(i, i) for i in range(3)], [(i, 2 - i) for i in range(3)]]

def ganador(t):
    for linea in LINEAS:
        valores = {t[f][c] for f, c in linea}
        if len(valores) == 1 and valores != {" "}:
            return valores.pop()
    return "empate" if all(c != " " for fila in t for c in fila) else None

def dos_jugadores():
    t, turno = [[" "] * 3 for _ in range(3)], "X"
    while ganador(t) is None:
        f, c = map(int, input(f"{turno} (fila columna): ").split())
        if t[f][c] != " ":
            print("Casilla ocupada")
            continue
        t[f][c] = turno
        turno = "O" if turno == "X" else "X"
    print("\n".join("|".join(fila) for fila in t))
    return ganador(t)

print("Resultado:", dos_jugadores())

X (fila columna): 1 1
O (fila columna): 0 0
X (fila columna): 0 1
O (fila columna): 2 2
X (fila columna): 2 1
O|X| 
 |X| 
 |X|O
Resultado: X


## 5. Tres en raya contra el ordenador

El ordenador elige su jugada con *minimax*: explora todas las partidas posibles y nunca pierde. Contra un rival perfecto el tres en raya acaba en empate, así que "ganar siempre" solo es posible si el rival comete errores.

In [5]:
from functools import cache

@cache
def minimax(estado, turno):
    t = [list(estado[i:i + 3]) for i in (0, 3, 6)]
    g = ganador(t)
    if g is not None:
        return {"X": 1, "O": -1, "empate": 0}[g], None
    mejores = []
    for i, casilla in enumerate(estado):
        if casilla == " ":
            valor, _ = minimax(estado[:i] + turno + estado[i + 1:], "O" if turno == "X" else "X")
            mejores.append((valor, i))
    return (max if turno == "X" else min)(mejores)

def partida(jugador_o, semilla=0):
    rng, estado, turno = random.Random(semilla), " " * 9, "X"
    while ganador([list(estado[i:i + 3]) for i in (0, 3, 6)]) is None:
        libres = [i for i, c in enumerate(estado) if c == " "]
        i = minimax(estado, "X")[1] if turno == "X" else (minimax(estado, "O")[1] if jugador_o == "perfecto" else rng.choice(libres))
        estado = estado[:i] + turno + estado[i + 1:]
        turno = "O" if turno == "X" else "X"
    return ganador([list(estado[i:i + 3]) for i in (0, 3, 6)])

print("Contra rival perfecto:", partida("perfecto"))
resultados = [partida("aleatorio", s) for s in range(200)]
print("Contra rival aleatorio:", {r: resultados.count(r) for r in set(resultados)})

Contra rival perfecto: empate
Contra rival aleatorio: {'empate': 1, 'X': 199}


## 6. Damas

Partida de damas para dos personas verificando los movimientos: los peones avanzan en diagonal una casilla, capturan saltando (la captura es obligatoria y puede encadenarse) y coronan como dama al llegar al final; la dama se mueve y captura a cualquier distancia en diagonal.

In [6]:
def tablero_damas():
    t = [["."] * 8 for _ in range(8)]
    for f in range(8):
        for c in range(8):
            if (f + c) % 2:
                t[f][c] = "n" if f < 3 else "b" if f > 4 else "."
    return t

def propias(pieza, color):
    return pieza.lower() == color

def capturas_desde(t, f, c, color):
    """Secuencias de captura (listas de casillas) desde (f, c)."""
    pieza, dama = t[f][c], t[f][c].isupper()
    secuencias = []
    for df, dc in ((1, 1), (1, -1), (-1, 1), (-1, -1)):
        if not dama and df != (1 if color == "n" else -1):
            continue
        paso, rf, rc = 1, f + df, c + dc
        while dama and 0 <= rf < 8 and 0 <= rc < 8 and t[rf][rc] == ".":
            rf, rc = rf + df, rc + dc
        if not (0 <= rf < 8 and 0 <= rc < 8) or t[rf][rc] == "." or propias(t[rf][rc], color):
            continue
        df2, dc2 = rf + df, rc + dc
        while 0 <= df2 < 8 and 0 <= dc2 < 8 and t[df2][dc2] == ".":
            # captura provisional para seguir encadenando
            capturada, t[rf][rc], t[f][c], t[df2][dc2] = t[rf][rc], ".", ".", pieza
            siguientes = capturas_desde(t, df2, dc2, color)
            t[rf][rc], t[f][c], t[df2][dc2] = capturada, pieza, "."
            secuencias += [[(df2, dc2)] + s for s in siguientes] or [[(df2, dc2)]]
            if not dama:
                break
            df2, dc2 = df2 + df, dc2 + dc
    return secuencias

def movimientos(t, color):
    capturas, simples = [], []
    for f in range(8):
        for c in range(8):
            if propias(t[f][c], color):
                capturas += [[(f, c)] + s for s in capturas_desde(t, f, c, color)]
                dama = t[f][c].isupper()
                for df, dc in ((1, 1), (1, -1), (-1, 1), (-1, -1)):
                    if not dama and df != (1 if color == "n" else -1):
                        continue
                    nf, nc = f + df, c + dc
                    while 0 <= nf < 8 and 0 <= nc < 8 and t[nf][nc] == ".":
                        simples.append([(f, c), (nf, nc)])
                        if not dama:
                            break
                        nf, nc = nf + df, nc + dc
    if capturas:
        larga = max(map(len, capturas))
        return [m for m in capturas if len(m) == larga]
    return simples

def aplica(t, mov, color):
    (f, c), pieza = mov[0], t[mov[0][0]][mov[0][1]]
    t[f][c] = "."
    for nf, nc in mov[1:]:
        df, dc = (nf > f) - (nf < f), (nc > c) - (nc < c)
        for k in range(1, abs(nf - f)):
            t[f + k * df][c + k * dc] = "."
        f, c = nf, nc
    if f == (7 if color == "n" else 0):
        pieza = pieza.upper()
    t[f][c] = pieza

t, color = tablero_damas(), "b"
rng = random.Random(4)
for turno in range(60):
    legales = movimientos(t, color)
    if not legales:
        print(f"{color} no puede mover: pierde")
        break
    aplica(t, rng.choice(legales), color)
    color = "n" if color == "b" else "b"
print("\n".join(" ".join(fila) for fila in t))
print("¿Mover un peón hacia atrás es legal?", [(5, 0), (6, 1)] in movimientos(tablero_damas(), "b"))

b no puede mover: pierde
. . . . . . . n
. . n . . . . .
. n . n . . . .
. . . . . . . .
. . . . . . . .
. . . . n . N .
. . . . . . . .
N . . . . . . .
¿Mover un peón hacia atrás es legal? False


## 7. Ajedrez

Partida de ajedrez para dos personas verificando que todos los movimientos son legales: movimientos de cada pieza, no dejar al propio rey en jaque, enroques, captura al paso y coronación (a dama). La corrección se comprueba con *perft*: el número de posiciones tras 3 jugadas desde el inicio es 8902.

In [7]:
INICIAL = ["rnbqkbnr", "pppppppp", "........", "........", "........", "........", "PPPPPPPP", "RNBQKBNR"]
DIRS = {"N": [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)],
        "B": [(1, 1), (1, -1), (-1, 1), (-1, -1)], "R": [(1, 0), (-1, 0), (0, 1), (0, -1)]}
DIRS["Q"] = DIRS["K"] = DIRS["B"] + DIRS["R"]

def nueva_partida():
    return {"t": [list(f) for f in INICIAL], "turno": "w", "enroques": set("KQkq"), "al_paso": None}

def color_de(p):
    return None if p == "." else "w" if p.isupper() else "b"

def atacada(t, f, c, por):
    for (pf, pc), p in ((x, t[x[0]][x[1]]) for x in [(i, j) for i in range(8) for j in range(8)]):
        if color_de(p) != por:
            continue
        tipo = p.upper()
        if tipo == "P":
            if f == pf + (-1 if por == "w" else 1) and abs(c - pc) == 1:
                return True
            continue
        for df, dc in DIRS[tipo]:
            nf, nc = pf + df, pc + dc
            while 0 <= nf < 8 and 0 <= nc < 8:
                if (nf, nc) == (f, c):
                    return True
                if t[nf][nc] != "." or tipo in "NK":
                    break
                nf, nc = nf + df, nc + dc
    return False

def pseudo(estado):
    t, yo = estado["t"], estado["turno"]
    for f in range(8):
        for c in range(8):
            p = t[f][c]
            if color_de(p) != yo:
                continue
            tipo = p.upper()
            if tipo == "P":
                d, inicio, final = (-1, 6, 0) if yo == "w" else (1, 1, 7)
                if 0 <= f + d < 8 and t[f + d][c] == ".":
                    yield (f, c, f + d, c)
                    if f == inicio and t[f + 2 * d][c] == ".":
                        yield (f, c, f + 2 * d, c)
                for dc in (-1, 1):
                    nf, nc = f + d, c + dc
                    if 0 <= nc < 8 and 0 <= nf < 8 and (color_de(t[nf][nc]) not in (None, yo) or estado["al_paso"] == (nf, nc)):
                        yield (f, c, nf, nc)
                continue
            for df, dc in DIRS[tipo]:
                nf, nc = f + df, c + dc
                while 0 <= nf < 8 and 0 <= nc < 8 and color_de(t[nf][nc]) != yo:
                    yield (f, c, nf, nc)
                    if t[nf][nc] != "." or tipo in "NK":
                        break
                    nf, nc = nf + df, nc + dc
    fila, rival = (7, "b") if yo == "w" else (0, "w")
    for lado, cols_vacias, cols_seguras in (("K", (5, 6), (4, 5, 6)), ("Q", (1, 2, 3), (4, 3, 2))):
        clave = lado if yo == "w" else lado.lower()
        if clave in estado["enroques"] and all(t[fila][x] == "." for x in cols_vacias) \
                and not any(atacada(t, fila, x, rival) for x in cols_seguras):
            yield (fila, 4, fila, 6 if lado == "K" else 2)

def mueve(estado, m):
    f, c, nf, nc = m
    t = [fila[:] for fila in estado["t"]]
    p = t[f][c]
    if p.upper() == "P" and (nf, nc) == estado["al_paso"]:
        t[f][nc] = "."
    if p.upper() == "K" and abs(nc - c) == 2:
        torre_desde, torre_hasta = (7, 5) if nc == 6 else (0, 3)
        t[f][torre_hasta], t[f][torre_desde] = t[f][torre_desde], "."
    t[nf][nc], t[f][c] = p, "."
    if p.upper() == "P" and nf in (0, 7):
        t[nf][nc] = "Q" if p.isupper() else "q"
    enroques = set(estado["enroques"])
    for (rf, rc), letras in (((7, 4), "KQ"), ((0, 4), "kq"), ((7, 7), "K"), ((7, 0), "Q"), ((0, 7), "k"), ((0, 0), "q")):
        if (f, c) == (rf, rc) or (nf, nc) == (rf, rc):
            enroques -= set(letras)
    al_paso = ((f + nf) // 2, c) if p.upper() == "P" and abs(nf - f) == 2 else None
    return {"t": t, "turno": "b" if estado["turno"] == "w" else "w", "enroques": enroques, "al_paso": al_paso}

def legales(estado):
    yo, rival = estado["turno"], "b" if estado["turno"] == "w" else "w"
    resultado = []
    for m in pseudo(estado):
        nuevo = mueve(estado, m)
        rey = next((f, c) for f in range(8) for c in range(8) if nuevo["t"][f][c] == ("K" if yo == "w" else "k"))
        if not atacada(nuevo["t"], *rey, rival):
            resultado.append(m)
    return resultado

def perft(estado, profundidad):
    if profundidad == 0:
        return 1
    return sum(perft(mueve(estado, m), profundidad - 1) for m in legales(estado))

def de_algebraica(jugada):
    c, f, nc, nf = jugada
    return (8 - int(f), "abcdefgh".index(c), 8 - int(nf), "abcdefgh".index(nc))

print([perft(nueva_partida(), d) for d in (1, 2, 3)])
partida = nueva_partida()
for jugada in ["f2f3", "e7e5", "e2e5", "g2g4", "d8h4"]:
    m = de_algebraica(jugada)
    if m not in legales(partida):
        print(jugada, "ilegal")
        continue
    partida = mueve(partida, m)
print("Jugadas legales de las blancas:", len(legales(partida)), "-> jaque mate (mate del loco)")

[20, 400, 8902]
e2e5 ilegal
Jugadas legales de las blancas: 0 -> jaque mate (mate del loco)
